# Modelos de Lenguaje — Armemos un Chat a Mano (Mini Modelo y RAG)

[![francomansilla.com](https://img.shields.io/badge/francomansilla.com-555555?style=flat&logo=globe&logoColor=white)](https://www.francomansilla.com)

Este notebook arma, paso a paso y sin entrenar nada, un mini modelo de lenguaje con 30 oraciones: convierte palabras en vectores (embeddings con IDF), aplica atención y predice la siguiente palabra con temperatura. Al final muestra, en miniatura, cómo funciona RAG: buscar en los textos antes de responder.

**Librerías**

[![NumPy](https://img.shields.io/badge/NumPy-013243?style=flat&logo=numpy&logoColor=white)](https://numpy.org/doc/)

**Modelos y métodos**

[![Embeddings](https://img.shields.io/badge/Embeddings-0078D4?style=flat&logo=python&logoColor=white)](https://es.wikipedia.org/wiki/Word_embedding)
[![IDF](https://img.shields.io/badge/IDF-6A0DAD?style=flat&logo=python&logoColor=white)](https://es.wikipedia.org/wiki/Tf-idf)
[![Atención](https://img.shields.io/badge/Atención-E34F26?style=flat&logo=python&logoColor=white)](https://arxiv.org/abs/1706.03762)
[![Softmax](https://img.shields.io/badge/Softmax-2E8B57?style=flat&logo=python&logoColor=white)](https://en.wikipedia.org/wiki/Softmax_function)
[![RAG](https://img.shields.io/badge/RAG-B8860B?style=flat&logo=python&logoColor=white)](https://arxiv.org/abs/2005.11401)

**Métricas**

[![Similitud coseno](https://img.shields.io/badge/Similitud_coseno-DC143C?style=flat&logo=python&logoColor=white)](https://es.wikipedia.org/wiki/Similitud_coseno)

---

In [1]:
import re
import numpy as np

# Mini Modelo de Lenguaje con 30 Oraciones

### El corpus
El **corpus** es todo el texto del que aprende el modelo. Aquí son las 30 oraciones de la planilla de la clase 2: 24 sobre mascotas y 6 sobre autos. Es todo lo que nuestro modelo «sabe» del mundo.

In [2]:
corpus = """
El perro es una mascota muy fiel.
El gato es una mascota independiente.
El conejo es una mascota tranquila.
Llevamos al perro al veterinario.
Llevamos al gato al veterinario.
Llevamos al conejo al veterinario.
El perro vive en mi casa.
El gato vive en mi casa.
El conejo vive en una jaula en el patio.
Le compré comida especial al perro.
Le compré comida especial al gato.
Le compré comida especial al conejo.
El perro ladra cuando llega una visita.
El gato maúlla cuando tiene hambre.
El conejo es un animal muy silencioso.
Saco a pasear al perro con correa todos los días.
El gato se trepa a los muebles y araña el sofá.
El conejo salta dentro de su jaula.
Mi perro mueve la cola cuando me ve llegar.
Mi gato ronronea cuando lo acaricio.
Mi conejo come zanahoria todas las tardes.
El perro y el gato duermen juntos en el sillón.
El conejo tiene las orejas muy largas.
Tanto el perro como el gato y el conejo necesitan agua fresca todos los días.
El auto necesita gasolina para funcionar.
Guardo el auto en la casa.
Mi papá maneja el auto todos los días.
El auto tiene cuatro ruedas nuevas.
El motor del auto falló esta mañana.
Estaciono el auto en el garaje todas las noches.
""".strip().split("\n")

print(len(corpus), "oraciones")

30 oraciones


### Separar en palabras
El computador no ve frases, ve listas de palabras. Pasamos todo a minúsculas, quitamos la puntuación y agregamos `<fin>` al final de cada oración, para que el modelo aprenda cuándo terminar.

Ejemplo: «El perro es una mascota muy fiel.» → `['el', 'perro', 'es', 'una', 'mascota', 'muy', 'fiel', '<fin>']`

In [3]:
def separar(texto):
    return re.findall(r"[a-záéíóúñü]+", texto.lower())

oraciones = [separar(o) + ["<fin>"] for o in corpus]
vocabulario = sorted({p for o in oraciones for p in o})
posicion = {p: i for i, p in enumerate(vocabulario)}   # palabra -> número de fila

print(oraciones[0])
print(vocabulario[0:])
print(posicion)
print("Vocabulario:", len(vocabulario), "palabras distintas")

['el', 'perro', 'es', 'una', 'mascota', 'muy', 'fiel', '<fin>']
['<fin>', 'a', 'acaricio', 'agua', 'al', 'animal', 'araña', 'auto', 'casa', 'cola', 'come', 'comida', 'como', 'compré', 'con', 'conejo', 'correa', 'cuando', 'cuatro', 'de', 'del', 'dentro', 'duermen', 'días', 'el', 'en', 'es', 'especial', 'esta', 'estaciono', 'falló', 'fiel', 'fresca', 'funcionar', 'garaje', 'gasolina', 'gato', 'guardo', 'hambre', 'independiente', 'jaula', 'juntos', 'la', 'ladra', 'largas', 'las', 'le', 'llega', 'llegar', 'llevamos', 'lo', 'los', 'maneja', 'mascota', 'mañana', 'maúlla', 'me', 'mi', 'motor', 'muebles', 'mueve', 'muy', 'necesita', 'necesitan', 'noches', 'nuevas', 'orejas', 'papá', 'para', 'pasear', 'patio', 'perro', 'ronronea', 'ruedas', 'saco', 'salta', 'se', 'silencioso', 'sillón', 'sofá', 'su', 'tanto', 'tardes', 'tiene', 'todas', 'todos', 'tranquila', 'trepa', 'un', 'una', 've', 'veterinario', 'visita', 'vive', 'y', 'zanahoria']
{'<fin>': 0, 'a': 1, 'acaricio': 2, 'agua': 3, 'al': 4, 'an

### Embeddings por coocurrencia — mejora con IDF
**Dime con quién andas y te diré quién eres.** El vector de cada palabra cuenta con qué otras palabras comparte oración. A diferencia de la planilla, usamos como columnas las 96 palabras del vocabulario, no solo 20.

**El problema:** «el» aparece en 20 de las 30 oraciones. Perro y auto parecen parecidos solo porque los dos van con «el». Es como decir que dos personas se parecen porque ambas tienen dos ojos.

**La mejora de la hoja 4: IDF** (*Inverse Document Frequency*, frecuencia inversa). Cada palabra recibe un peso: cuanto más rara, más pesa.

**IDF(palabra) = log( total de oraciones / oraciones donde aparece )**

| Palabra | Aparece en | IDF |
|---|---|---|
| el | 20 de 30 | 0,41 → casi no cuenta |
| veterinario | 3 de 30 | 2,30 |
| gasolina | 1 de 30 | 3,40 → cuenta mucho |

Así, compartir «el» casi no suma y compartir «veterinario» suma mucho. Resultado: coseno(perro, auto) baja de 0,52 a 0,17. El logaritmo suaviza los pesos: sin él, «gasolina» pesaría 30 contra 1,5 de «el».

In [4]:
N = len(vocabulario)
conteos = np.zeros((N, N))
for o in oraciones:
    palabras = set(o) - {"<fin>"}
    for a in palabras:
        for b in palabras:
            if a != b:
                conteos[posicion[a], posicion[b]] += 1

# IDF: log(total de oraciones / oraciones donde aparece la palabra)
aparece_en = np.array([sum(p in o for o in oraciones) for p in vocabulario])
idf = np.log(len(oraciones) / aparece_en)
embeddings = conteos * idf          # cada fila es el vector de una palabra

def emb(palabra):
    return embeddings[posicion[palabra]]

def coseno(a, b):
    """La misma fórmula de las hojas 1 y 3: (A·B) / (|A| × |B|)."""
    largo = np.linalg.norm(a) * np.linalg.norm(b)
    return 0.0 if largo == 0 else float(a @ b / largo)

for a, b in [("perro", "gato"), ("perro", "auto"), ("zanahoria", "conejo"), ("zanahoria", "auto")]:
    print(f"coseno({a}, {b}) = {coseno(emb(a), emb(b)):.2f}")

coseno(perro, gato) = 0.63
coseno(perro, auto) = 0.17
coseno(zanahoria, conejo) = 0.34
coseno(zanahoria, auto) = 0.15


### Atención — a qué palabras mira cada una
Cada palabra «mira» a las demás de la oración y reparte su atención: más a las que se le parecen, menos a las que no. Como en una conversación: si alguien dice «perro», piensas más en «gato» que en «sillón».

Es la receta de la clase 3: **coseno × escala → softmax → promedio ponderado**. Ejemplo con «perro» en *el perro y el gato duermen juntos en el sillón*:

| Palabra | Coseno con «perro» | × 5 (escala) | Softmax = atención |
|---|---|---|---|
| perro | 1,00 | 5,00 | 78,8 % |
| gato | 0,63 | 3,17 | 12,7 % |
| duermen | 0,34 | 1,68 | 2,8 % |
| juntos | 0,34 | 1,68 | 2,8 % |
| sillón | 0,34 | 1,68 | 2,8 % |

- **Coseno:** cuánto se parece cada palabra a «perro».
- **× escala:** agranda las diferencias para que la atención se concentre en las más parecidas.
- **Softmax:** convierte los puntajes en porcentajes que suman 100 %.
- **Promedio ponderado:** el nuevo vector de «perro» es 78,8 % de perro + 12,7 % de gato + un poco del resto.

**La fórmula de softmax.** A cada puntaje le aplica la exponencial (eˣ, con e ≈ 2,718) y divide por la suma de todos:

$$\text{softmax}(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

| Palabra | Puntaje | e^puntaje | ÷ suma |
|---|---|---|---|
| perro | 5,00 | 148,41 | 148,41 / 188,27 = **78,8 %** |
| gato | 3,17 | 23,83 | 23,83 / 188,27 = 12,7 % |
| duermen, juntos, sillón | 1,68 | 5,34 c/u | 5,34 / 188,27 = 2,8 % c/u |
| **Suma** | | **188,27** | 100 % |

La exponencial siempre da números positivos y agranda las diferencias: perro y gato están a menos del doble en puntaje (5,00 vs. 3,17), pero a más de 6 veces después de eˣ (148 vs. 24). Por eso la atención se concentra en las palabras más parecidas.

**¿Por qué «perro» se mira a sí mismo un 78,8 %?** Toda palabra es idéntica a sí misma (coseno = 1), así que siempre queda primera. Lo interesante es el segundo lugar: «perro» mira a «gato», la palabra más parecida, y su vector absorbe un poco de «gato». En los modelos reales las palabras pasan antes por transformaciones aprendidas, así que pueden mirar más a otras que a sí mismas.

Simplificación: dejamos fuera las «palabras vacías» (el, en, una, mi…). Un modelo real aprende solo a ignorarlas; aquí se lo decimos a mano.

In [5]:
VACIAS = {"el", "la", "los", "las", "un", "una", "en", "al", "a", "de", "del", "y", "mi", "su",
          "lo", "le", "me", "se", "con", "para", "cuando", "muy", "como", "tanto", "es", "<fin>"}

def softmax(puntajes):
    e = np.exp(puntajes - np.max(puntajes))
    return e / e.sum()

def atencion(consulta, palabras, escala=5):
    """Cuánta atención le pone 'consulta' a cada palabra, y el vector resultante."""
    puntajes = np.array([coseno(emb(consulta), emb(p)) for p in palabras]) * escala
    pesos = softmax(puntajes)
    vector = sum(w * emb(p) for w, p in zip(pesos, palabras))
    return pesos, vector

frase = separar("el perro y el gato duermen juntos en el sillón")
importantes = [p for p in frase if p not in VACIAS]
pesos, _ = atencion("perro", importantes)
print("¿A quién mira «perro» en esta oración?")
for p, w in zip(importantes, pesos):
    print(f"  {p:8s} {w:5.1%}  " + "█" * int(w * 40))

¿A quién mira «perro» en esta oración?
  perro    78.8%  ███████████████████████████████
  gato     12.7%  █████
  duermen   2.8%  █
  juntos    2.8%  █
  sillón    2.8%  █


### Capa de atención — un vector para toda la oración
Arriba vimos la atención de **una** palabra. Para resumir la oración completa:

1. Cada palabra importante hace su propia atención y obtiene su vector mezclado.
2. Promediamos esos vectores. El resultado es el **contexto**: un solo vector que dice de qué habla la oración.

Eso es, en miniatura, una **capa de atención**.

Ejemplo: en *el auto tiene* las palabras importantes son «auto» y «tiene», y su promedio es el contexto. Abajo comparamos el contexto de dos oraciones con «hambre» y con «cuatro» (coseno más alto = se parece más):

| Oración | Coseno con «hambre» | Coseno con «cuatro» | Suena más a… |
|---|---|---|---|
| mi perro mueve la cola cuando tiene | 0,24 | 0,14 | hambre |
| el auto tiene | 0,23 | 0,46 | cuatro |

In [6]:
def contexto(palabras, escala=5):
    importantes = [p for p in palabras if p not in VACIAS] or palabras
    vectores = [atencion(p, importantes, escala)[1] for p in importantes]
    return np.mean(vectores, axis=0)

# ¿Se cumple? (como en la hoja 6): ¿el contexto se parece más a «hambre» o a «cuatro»?
for f in ["mi perro mueve la cola cuando tiene", "el auto tiene"]:
    ctx = contexto(separar(f))
    print(f"{f:38s} hambre: {coseno(emb('hambre'), ctx):.2f}   cuatro: {coseno(emb('cuatro'), ctx):.2f}")

mi perro mueve la cola cuando tiene    hambre: 0.24   cuatro: 0.14
el auto tiene                          hambre: 0.23   cuatro: 0.46


### Orden de las palabras — qué viene después
Los embeddings saben qué palabras van juntas, pero no en qué orden: para ellos «el perro ladra» y «ladra perro el» son lo mismo. Por eso contamos **qué palabra viene justo después de cuál** en el corpus.

Ejemplo: «tiene» aparece en tres oraciones del corpus:
- El gato maúlla cuando tiene **hambre**.
- El conejo tiene **las** orejas muy largas.
- El auto tiene **cuatro** ruedas nuevas.

Por eso abajo sale `{'cuatro': 1, 'hambre': 1, 'las': 1}`: cada una vino **1 vez** después de «tiene». En probabilidades, cada una tiene 1/3 (33 %). Solo con el orden, el modelo no puede elegir entre ellas.

In [7]:
siguientes = np.zeros((N, N))
for o in oraciones:
    for actual, despues in zip(o, o[1:]):
        siguientes[posicion[actual], posicion[despues]] += 1

def que_sigue(palabra):
    fila = siguientes[posicion[palabra]]
    return {vocabulario[i]: int(c) for i, c in enumerate(fila) if c > 0}

print("Después de «tiene»:", que_sigue("tiene"))
print("Después de «cuando»:", que_sigue("cuando"))

Después de «tiene»: {'cuatro': 1, 'hambre': 1, 'las': 1}
Después de «cuando»: {'llega': 1, 'lo': 1, 'me': 1, 'tiene': 1}


### Predecir la siguiente palabra — orden + tema
Cada candidata recibe un puntaje con dos partes:

**puntaje = log(probabilidad según el orden) + β × coseno(candidata, contexto)**

- **Orden:** ¿qué tan seguido vino esta palabra después de la última? Sale de «Orden de las palabras».
- **Tema:** ¿encaja con lo que habla la oración? Sale de la atención.
- **β** decide cuánto pesa el tema. Con β = 0 el modelo solo mira la última palabra.

Al final aplicamos softmax con **temperatura**, igual que en el simulador de la clase 1.

**Ejemplo: «el auto tiene …»** con β = 6

| Candidata | Orden: log(1/3) | Tema: coseno con el contexto | Puntaje = orden + 6 × tema | Probabilidad |
|---|---|---|---|---|
| cuatro | −1,10 | 0,46 | 1,69 | 76 % |
| hambre | −1,10 | 0,23 | 0,31 | 19 % |
| las | −1,10 | 0 (palabra vacía) | −1,10 | 5 % |

Por orden, las tres empatan. El tema rompe el empate: «cuatro» encaja con auto y gana. El log solo convierte la probabilidad en puntaje: más probable, puntaje más alto.

In [8]:
def probabilidades(palabras, beta=6, temperatura=1.0, escala=5):
    fila = siguientes[posicion[palabras[-1]]]
    candidatas = [vocabulario[i] for i in np.where(fila > 0)[0]]
    p_orden = np.array([fila[posicion[c]] for c in candidatas]) / fila.sum()

    ctx = contexto(palabras, escala)
    encaje = np.array([0.0 if c in VACIAS else coseno(emb(c), ctx) for c in candidatas])

    puntaje = np.log(p_orden) + beta * encaje
    return candidatas, softmax(puntaje / temperatura)

def mostrar(texto, **opciones):
    candidatas, probs = probabilidades(separar(texto), **opciones)
    print(f"«{texto} …»   {opciones}")
    for i in np.argsort(-probs)[:5]:
        print(f"  {candidatas[i]:10s} {probs[i]:5.1%}  " + "█" * int(probs[i] * 40))
    print()

### Experimento central — con y sin atención
Las dos frases terminan en «tiene», y en el corpus a «tiene» le siguen «cuatro», «hambre» o «las».

- Con **β = 0** el modelo solo ve «tiene»: da 33 % a cada una y no distingue las frases.
- Con **β = 6** usa el tema: con perro gana «hambre»; con auto gana «cuatro».

Es el mismo efecto del jaguar en la clase 3.

In [9]:
mostrar("mi perro mueve la cola cuando tiene", beta=0)
mostrar("el auto tiene", beta=0)
mostrar("mi perro mueve la cola cuando tiene", beta=6)
mostrar("el auto tiene", beta=6)

«mi perro mueve la cola cuando tiene …»   {'beta': 0}
  cuatro     33.3%  █████████████
  hambre     33.3%  █████████████
  las        33.3%  █████████████

«el auto tiene …»   {'beta': 0}
  cuatro     33.3%  █████████████
  hambre     33.3%  █████████████
  las        33.3%  █████████████

«mi perro mueve la cola cuando tiene …»   {'beta': 6}
  hambre     54.9%  █████████████████████
  cuatro     31.7%  ████████████
  las        13.3%  █████

«el auto tiene …»   {'beta': 6}
  cuatro     76.1%  ██████████████████████████████
  hambre     19.2%  ███████
  las         4.7%  █



### Escribir texto — repetir la predicción
Predecimos, sorteamos una palabra según sus probabilidades, la pegamos al final y repetimos hasta llegar a `<fin>`. Es como apretar una y otra vez la sugerencia del teclado del celular. **Esto es todo lo que hace un modelo de lenguaje al escribir.**

In [10]:
azar = np.random.default_rng(2)   # semilla fija: todos obtienen lo mismo

def escribir(inicio, max_palabras=15, **opciones):
    palabras = separar(inicio)
    for _ in range(max_palabras):
        candidatas, probs = probabilidades(palabras, **opciones)
        elegida = azar.choice(candidatas, p=probs)
        if elegida == "<fin>":
            break
        palabras.append(elegida)
    return " ".join(palabras)

for inicio in ["el gato", "mi perro", "el conejo", "el auto"]:
    print(escribir(inicio))

el gato duermen juntos en el gato maúlla cuando llega una mascota tranquila
mi perro mueve la cola cuando tiene hambre
el conejo come zanahoria todas las tardes
el auto todos los días


### RAG en miniatura — buscar antes de responder
Hasta ahora el modelo solo usa las palabras que le escribimos. **RAG** (generación aumentada con recuperación) agrega un paso antes de responder: **buscar** en una base de textos la información relevante y **pegarla** como contexto. Es como una prueba con libro abierto. Los chats reales lo usan para responder con documentos propios (manuales, precios, políticas) sin reentrenar el modelo.

Usa solo piezas que ya vimos:

| Paso | Qué hace | Pieza que ya vimos |
|---|---|---|
| 1. Buscar | Encuentra la oración del corpus más parecida a la pregunta | Capa de atención (un vector por oración) + coseno |
| 2. Pegar | Agrega esa oración al contexto | Contexto de «Predecir la siguiente palabra» |
| 3. Responder | Predice la siguiente palabra | `probabilidades` |

**Ejemplo:** «¿Qué tiene el que maúlla?». La respuesta empieza con «tiene», y ya sabemos que después de «tiene» puede venir «cuatro», «hambre» o «las».

| | Lo que lee el modelo | cuatro | hambre | |
|---|---|---|---|---|
| Sin RAG | solo la pregunta: con «maúlla» como única pista no alcanza | 53 % | 44 % | se equivoca |
| Con RAG | la pregunta + «El gato maúlla cuando tiene hambre.» (coseno 0,68) | 28 % | 69 % | acierta |

El modelo no aprendió nada nuevo: solo cambió lo que **leyó** antes de responder. En un chat real la base son documentos y el modelo es mucho más grande, pero los pasos son los mismos.

In [11]:
def conocidas(texto):
    """Palabras del texto que están en el vocabulario del modelo."""
    return [p for p in separar(texto) if p in posicion]

def buscar(pregunta):
    """Paso 1: la oración del corpus más parecida a la pregunta."""
    q = contexto(conocidas(pregunta))                       # vector de la pregunta (capa de atención)
    puntajes = [coseno(q, contexto(o)) for o in oraciones]  # coseno con cada oración del corpus
    i = int(np.argmax(puntajes))
    print(f"Encontrada ({puntajes[i]:.2f}): {corpus[i]}\n")
    return separar(corpus[i])

pregunta = "¿Qué tiene el que maúlla?"
pegado = buscar(pregunta)                                   # pasos 1 y 2: buscar y pegar

for nombre, palabras in [("Sin RAG", conocidas(pregunta)), ("Con RAG", pegado + conocidas(pregunta))]:
    candidatas, probs = probabilidades(palabras + ["tiene"])  # paso 3: la respuesta empieza con «tiene»
    print(f"{nombre}: " + "   ".join(f"{c} {p:.0%}" for c, p in zip(candidatas, probs)))

Encontrada (0.68): El gato maúlla cuando tiene hambre.

Sin RAG: cuatro 53%   hambre 44%   las 3%
Con RAG: cuatro 28%   hambre 69%   las 3%


### Para jugar
1. **Sin atención:** usa `beta=0` y genera varias veces. ¿Aparecen frases como «mi perro mueve la cola cuando tiene cuatro ruedas nuevas»?
2. **Temperatura:** prueba `temperatura=0.3` y `temperatura=3`. ¿Cuál repite más el corpus? ¿Cuál inventa más?
3. **Más datos:** agrega oraciones al corpus y vuelve a ejecutar todo desde «El corpus».
4. **RAG:** cambia la pregunta, por ejemplo «¿Qué tiene el que ronronea?». ¿Qué oración encuentra? ¿Cambia la respuesta?

**Lo que no sabe hacer:** a veces escribe «el conejo vive en el conejo». Sabe qué palabras se relacionan, pero no tiene gramática ni sentido común. Los modelos reales los consiguen con miles de millones de oraciones y entrenamiento.

In [12]:
# 1. Sin atención: beta=0
print("— beta=0 —")
azar = np.random.default_rng(10)
for _ in range(4):
    print(escribir("mi perro mueve la cola cuando", beta=0))

# 2. Temperatura: cambia los valores y vuelve a ejecutar
for t in [0.3, 3]:
    print(f"\n— temperatura={t} —")
    azar = np.random.default_rng(10)
    for _ in range(4):
        print(escribir("el gato", temperatura=t))

# 3. Más datos: agrega oraciones en la celda «El corpus» y ejecuta todo de nuevo
# 4. RAG: cambia la pregunta en la celda «RAG en miniatura» y vuelve a ejecutarla

— beta=0 —
mi perro mueve la cola cuando tiene cuatro ruedas nuevas
mi perro mueve la cola cuando llega una visita
mi perro mueve la cola cuando tiene las orejas muy silencioso
mi perro mueve la cola cuando tiene cuatro ruedas nuevas

— temperatura=0.3 —
el gato ronronea cuando tiene hambre
el gato duermen juntos en el gato duermen juntos en el gato duermen juntos en el gato
el gato duermen juntos en mi gato duermen juntos en el gato duermen juntos en el gato
el gato duermen juntos en el gato duermen juntos en el gato duermen juntos en el gato

— temperatura=3 —
el gato y araña el conejo necesitan agua fresca todos los muebles y araña el perro vive
el gato y araña el conejo vive en una jaula en la casa
el gato ronronea cuando me ve llegar
el gato ronronea cuando llega una mascota independiente


# Cierre — Tres Ideas

1. **Un modelo de lenguaje predice la siguiente palabra**, una y otra vez. Los modelos reales hacen lo mismo, a otra escala.
2. **La atención le da tema a la predicción:** con la misma última palabra («tiene»), el resto de la oración decide si sigue «hambre» o «cuatro».
3. **RAG no le enseña nada al modelo:** busca el texto relevante y lo pega como contexto. Por eso todo depende de que la búsqueda encuentre la oración correcta.